In [ ]:
# slide 7 (repo secret)

name: Repository Secret Demo
on:
  push:
    branches: [ main ]
jobs:
  demo:
    runs-on: ubuntu-latest
    steps:
      - name: Use repository secret
        run: |
          echo "Repo secret is available!"
          echo "Masked value: $MY_SECRET"
        env:
          MY_SECRET: ${{ secrets.MY_SECRET }}




npx prettier --write .github/workflows/repo-secret.yml

actionlint

In [ ]:
# slide 8 (org secret)
# create org, secret and public repo inside the ord

name: Org Secret Demo
on:
  push:
    branches: [main]
jobs:
  demo:
    runs-on: ubuntu-latest
    steps:
      - name: Use organization secret
        run: |
          echo "Organization secret works!"
          echo "Masked value: $ORG_API_KEY"
        env:
          ORG_API_KEY: ${{ secrets.ORG_API_KEY }}

npx prettier --write .github/workflows/org-secret.yml

actionlint

In [ ]:
# slide 8 (env secret)

name: Environment Secret Demo
on:
  push:
    branches: [main]
jobs:
  prod:
    runs-on: ubuntu-latest
    environment: production # ← this env HAS the secret
    steps:
      - run: echo "Production secret $PROD_DB_PASSWORD"
        env:
          PROD_DB_PASSWORD: ${{ secrets.PROD_DB_PASSWORD }}
  staging:
    runs-on: ubuntu-latest
    environment: staging # ← this env does NOT have the secret
    steps:
      - run: echo "Production secret $PROD_DB_PASSWORD"
        env:
          PROD_DB_PASSWORD: ${{ secrets.PROD_DB_PASSWORD }}



npx prettier --write .github/workflows/9_env_secret.yml

actionlint

In [ ]:

name: Test OIDC with AWS
on:
  push:
    branches: [main]
permissions:
  id-token: write
  contents: read
jobs:
  aws-test:
    runs-on: ubuntu-latest
    steps:
      - name: Print OIDC sub/aud
        run: |
          curl -s -H "Authorization: bearer $ACTIONS_ID_TOKEN_REQUEST_TOKEN" \
            "$ACTIONS_ID_TOKEN_REQUEST_URL&audience=sts.amazonaws.com" \
          | python3 -c "import sys,json,base64; t=json.load(sys.stdin)['value'].split('.')[1]; t+='='*(-len(t)%4); print(base64.urlsafe_b64decode(t).decode())"
      - uses: actions/checkout@v5
      - name: Configure AWS credentials
        uses: aws-actions/configure-aws-credentials@v6
        with:
          role-to-assume: arn:aws:iam::767397813467:role/github-actions
          aws-region: us-east-1
      - name: Who am I?
        run: aws sts get-caller-identity
      - name: List files in zt-contract bucket
        run: aws s3 ls s3://zt-contract/






npx prettier --write .github/workflows/oidc-test.yml

actionlint


# https://github.blog/changelog/2026-04-23-immutable-subject-claims-for-github-actions-oidc-tokens/

In [ ]:
# slide 15

name: Python Build
on:
  workflow_call:
    inputs:
      python-version:
        type: string
        default: "3.11"
jobs:
  build:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v5
      - uses: actions/setup-python@v6
        with:
          python-version: ${{ inputs.python-version }}
      - run: |
          python -m pip install --upgrade pip
          pip install -r requirements.txt
      - run: pytest tests/test_unit.py -q # unit tests



npx prettier --write .github/workflows/python-build.yml

actionlint

----

pytest
requests

requirements.txt

----

def test_addition():
    assert 1 + 1 == 2

def test_uppercase():
    assert "data".upper() == "DATA"

tests/test_unit.py

----


name: CI with E2E
on: [push, pull_request]
jobs:
  build:
    uses: ./.github/workflows/python-build.yml   # calls the reusable workflow
    with:
      python-version: "3.11"
  e2e:
    needs: build                                 # only runs if build passed
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v5
      - uses: actions/setup-python@v6
        with:
          python-version: "3.11"
      - run: |
          python -m pip install --upgrade pip
          pip install -r requirements.txt
      - run: pytest tests/e2e -q                 # e2e tests





npx prettier --write .github/workflows/ci.yml

actionlint



----



import requests

def test_github_api_up():
    resp = requests.get("https://api.github.com", timeout=10)
    assert resp.status_code == 200

tests/e2e/test_e2e.py

In [ ]:
# slide 19
# just change the ci.yml


name: CI — matrix builds + E2E
on: [push, pull_request]
jobs:
  build:
    strategy:
      matrix:
        python-version: ["3.10", "3.11"]   # ← one job per version
      fail-fast: false                      # ← see every combo, don't cancel on first fail
    uses: ./.github/workflows/python-build.yml
    with:
      python-version: ${{ matrix.python-version }}

  e2e:
    needs: build          # ← waits for ALL matrix runs to finish
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v5
      - uses: actions/setup-python@v6
        with:
          python-version: "3.11"
      - run: |
          python -m pip install --upgrade pip
          pip install -r requirements.txt   # ← REQUIRED, or the e2e test can't import requests
      - run: pytest tests/e2e -q



npx prettier --write .github/workflows/ci.yml

actionlint